# Run All Paper Experiments

This notebook runs all experiments from the paper in optimal order:
- Groups experiments by base model to reuse trained obfuscation embeddings
- Saves checkpoints so re-running skips completed embedding training
- Collects all results into a single summary

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "vit_obfuscation").exists():
            return path
    raise RuntimeError("Could not find repository root. Start Jupyter inside the repository.")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

CONFIG_DIR = ROOT / "configs" / "experiments"
CANONICAL_OUTPUT_DIR = ROOT / "outputs" / "revision_v3"
NOTEBOOK_OUTPUT_DIR = ROOT / "outputs" / "notebook_runs"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Canonical artifacts: {CANONICAL_OUTPUT_DIR}")
print(f"Notebook outputs: {NOTEBOOK_OUTPUT_DIR}")

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
for name in ["httpx", "httpcore", "filelock", "huggingface_hub"]:
    logging.getLogger(name).setLevel(logging.WARNING)

## Run All Experiments

The `run_all_experiments` function:
1. Scans `configs/experiments/` for YAML configs
2. Groups by base model (e.g., all ViT experiments together)
3. Trains the obfuscation embedding once per model, caches it
4. Runs each experiment, saving individual + combined results

In [ ]:
from vit_obfuscation.runner.runner import run_all_experiments

# Writes fresh notebook runs outside the frozen manuscript artifact directory.
all_results = run_all_experiments(
    config_dir=str(CONFIG_DIR),
    output_dir=str(NOTEBOOK_OUTPUT_DIR),
)

## Results Summary

In [ ]:
import pandas as pd

rows = []
for r in all_results:
    if "error" in r:
        rows.append({"experiment": r["experiment"], "status": "FAILED", "error": r["error"]})
    else:
        row = {"experiment": r["experiment"], "model": r["model"], "task": r["task"], "status": "OK"}
        for k, v in r.get("clean", {}).items():
            row[f"clean_{k}"] = v
        for k, v in r.get("obfuscated", {}).items():
            row[f"obf_{k}"] = v
        rows.append(row)

df = pd.DataFrame(rows)
df

In [ ]:
import json

print(json.dumps(all_results, indent=2, default=str))